# 6 (2.5 points)

Given the following information as of December 31, 2018:

In [1]:
import pandas as pd
import chainladder as cl

# Values in thousands ($000)
tri = cl.Triangle(
    data={
        'origin': [2016, 2016, 2016, 2017, 2017, 2018],
        'development': [2016, 2017, 2018, 2017, 2018, 2018],
        'Paid Loss': [360, 1425, 2850, 375, 1375, 350]
    },
    origin="origin",
    development="development",
    columns="Paid Loss",
    cumulative=True
)

olep = pd.Series([13000, 13250, 13500], index=[2016, 2017, 2018])

print(tri)
print(olep)

         12      24      36
2016  360.0  1425.0  2850.0
2017  375.0  1375.0     NaN
2018  350.0     NaN     NaN
2016    13000
2017    13250
2018    13500
dtype: int64


- The expected loss payment pattern follows a loglogistic curve of the form $\frac{x^\omega}{x^\omega + \theta^\omega}$, where
    - $\omega = 1.448$
    - $\theta = 48.021$
- There are no payments after 120 months.
- Accidents occur uniformly throughut the year.
- The scale parameter, $\sigma^2$, is 423.

* a. (1.75 points) Calculate the incremental fitted payment and corresponding normalized residual for accident year at 2018 at 12 months using the Cape Cod method.
* b. (0.75 point) Caluclate ultimate losses for the accident year 2016 using the Cape Cod method.

## Solution (Clark, *LDF Curve-Fitting and Stochastic Reserving*, Cape Cod method)

Clark's conventions (Section 2 / Appendix B), used below:
- The growth curve is evaluated at the **average accident date**, so an accident year at age 12, 24, 36 months uses $x=6, 18, 30$ (accidents occur uniformly, accident-year periods that are fully earned).
- Cape Cod ELR (MLE with $\omega,\theta$ fixed): $ELR=\sum_i C_i\ /\ \sum_i \text{Premium}_i\cdot G(x_i)$.
- Expected incremental loss from age $a$ to $b$: $\mu = ELR\cdot\text{Premium}\cdot[G(b-6)-G(a-6)]$, with $G(-6)\equiv0$.
- Normalized residual: $r=(c-\mu)/\sqrt{\sigma^2\mu}$.
- Truncation: the paper truncates at 240 months using $G(234)$, i.e. the average age is truncation age minus 6. By the same convention, "no payments after 120 months" means the ultimate uses $G(114)$.

In [2]:
import numpy as np

omega, theta, sigma2 = 1.448, 48.021, 423.0
G = lambda x: x**omega / (x**omega + theta**omega)

ages = np.array([12, 24, 36])
prem = olep.values.astype(float)                       # 2016, 2017, 2018
cum = tri.to_frame().values                            # origin x age, cumulative
latest = np.array([cum[i, ~np.isnan(cum[i])][-1] for i in range(3)])
latest_age = np.array([36, 24, 12])
G_latest = G(latest_age - 6)

elr = latest.sum() / (prem * G_latest).sum()
print("G at avg ages (2016, 2017, 2018):", G_latest.round(4))
print("ELR:", round(elr, 4))

# a. AY2018, 0-12 months
c_2018 = 350.0
mu_2018 = elr * prem[2] * (G(12 - 6) - 0)
resid_2018 = (c_2018 - mu_2018) / np.sqrt(sigma2 * mu_2018)
print(f"a. fitted incremental = {mu_2018:.1f}, normalized residual = {resid_2018:.4f}")

# b. AY2016 ultimate with truncation at 120 months (avg age 114)
ult_2016 = latest[0] + elr * prem[0] * (G(120 - 6) - G(36 - 6))
print(f"b. AY2016 ultimate = {ult_2016:,.1f}")
print(f"   (if G(120) were used instead: {latest[0] + elr * prem[0] * (G(120) - G(30)):,.1f})")

assert np.isclose(mu_2018, 382.2, atol=0.05) and np.isclose(resid_2018, -0.0802, atol=1e-3)
assert np.isclose(ult_2016, 6315.8, atol=0.05)

G at avg ages (2016, 2017, 2018): [0.336  0.1945 0.0469]
ELR: 0.6037
a. fitted incremental = 382.2, normalized residual = -0.0802
b. AY2016 ultimate = 6,315.8
   (if G(120) were used instead: 6,414.6)


**a.** $G(6)=0.0469$; $\sum C=2{,}850+1{,}375+350=4{,}575$ and $\sum \text{Prem}\cdot G(x)$ uses $G(30)=0.3360$, $G(18)=0.1945$, $G(6)=0.0469$, giving $ELR\approx60.4\%$.
Fitted incremental for AY 2018 at 12 months: $0.6037\times13{,}500\times0.0469\approx$ **382.2**.
Normalized residual: $(350-382.2)/\sqrt{423\times382.2}\approx$ **−0.080**.

**b.** AY 2016 ultimate $=2{,}850+ELR\times13{,}000\times[G(114)-G(30)]=2{,}850+0.6037\times13{,}000\times(0.7776-0.3360)\approx$ **6,316 ($000)**.

*Assumption flagged:* I used the paper's convention that truncation at age $T$ uses the average age $T-6$ (as in $G(234)$ for the 240-month truncation). If the intended answer used $G(120)$ directly, the ultimate would be ≈ 6,415 instead.

## chainladder-python gaps

- `cl.ClarkLDF(growth=...)` always **fits** $\omega,\theta$ by MLE. There is no way to supply known parameters. On this 3-year triangle it fits $\omega\approx1.30$, $\theta\approx1210$, nonsense for 6 data points, so the problem's given $\omega=1.448,\theta=48.021$ can't be used.
- No **scale parameter** input and no `truncation_age`/tail option inside `ClarkLDF` (only via `TailClark`, which fits its own curve).
- No method to **evaluate the fitted growth curve** $G(x)$ at an arbitrary age, or to return the **expected incremental losses** for each cell on the *given* parameters. `scale_` and `norm_resid_` are wrong for Cape Cod (see the bug in `2019_5.ipynb`: they use latest-diagonal ultimates instead of ELR × premium).
- No Appendix B handling of partly earned periods or policy-year periods.
- Suggested feature: `ClarkLDF(growth, omega=None, theta=None, scale=None, truncation_age=None)` where supplied parameters are held fixed, plus `.growth_curve(age)`, `.expected_incremental_` (cells), `.ultimate_` / `.ibnr_`, and the `elr_` computed from them.

## Sketch: `ClarkCapeCod` with fixed growth-curve parameters

A small prototype (not part of chainladder) to show the interface: parameters supplied or fitted, outputs for the expected incrementals, normalized residuals, ELR and ultimates.

In [3]:
from sklearn.base import BaseEstimator


class ClarkCapeCod(BaseEstimator):
    """Clark Cape Cod with known (omega, theta, scale); 12-month accident-year periods only."""

    def __init__(self, omega, theta, scale, truncation_age=None):
        self.omega, self.theta, self.scale, self.truncation_age = omega, theta, scale, truncation_age

    def growth_curve(self, x):
        x = np.asarray(x, dtype=float)
        return x**self.omega / (x**self.omega + self.theta**self.omega)

    def fit(self, X, y=None, sample_weight=None):
        cum = X.values[0, 0]
        prem = np.asarray(sample_weight, dtype=float)
        inc = np.diff(cum, axis=1, prepend=0)
        ages = np.array([int(a) for a in X.development])
        avg = np.maximum(ages - 6, 0)                       # average accident date
        Gk = np.where(avg > 0, self.growth_curve(np.maximum(avg, 1e-12)), 0.0)
        incG = np.diff(Gk, prepend=0)
        seen = ~np.isnan(inc)
        last = seen.sum(axis=1) - 1
        latest = np.array([cum[i, last[i]] for i in range(len(prem))])
        self.elr_ = latest.sum() / (prem * Gk[last]).sum()
        self.expected_incremental_ = self.elr_ * prem[:, None] * incG[None, :]
        self.norm_resid_ = (inc - self.expected_incremental_) / np.sqrt(self.scale * self.expected_incremental_)
        T = self.truncation_age or ages.max()
        G_T = float(self.growth_curve(T - 6))
        self.ibnr_ = self.elr_ * prem * (G_T - Gk[last])
        self.ultimate_ = latest + self.ibnr_
        return self


est = ClarkCapeCod(omega, theta, sigma2, truncation_age=120).fit(tri, sample_weight=prem)
print("ELR:", round(est.elr_, 4))
print("expected incremental:\n", pd.DataFrame(est.expected_incremental_, index=olep.index, columns=ages).round(1))
print("normalized residuals:\n", pd.DataFrame(est.norm_resid_, index=olep.index, columns=ages).round(3))
print("ultimate:", est.ultimate_.round(1))
assert np.isclose(est.expected_incremental_[2, 0], mu_2018) and np.isclose(est.ultimate_[0], ult_2016)

ELR: 0.6037
expected incremental:
          12      24      36
2016  368.1  1158.5  1110.2
2017  375.2  1180.8  1131.6
2018  382.2  1203.1  1152.9
normalized residuals:
         12     24     36
2016 -0.02 -0.134  0.459
2017 -0.00 -0.256    NaN
2018 -0.08    NaN    NaN
ultimate: [6315.8 6039.1 6305.1]


Extensions not prototyped: fitting $\omega,\theta$ when not supplied (the existing `ClarkLDF` machinery), the parameter covariance and variance outputs from `2019_5.ipynb`'s sketch, and Appendix B for partial periods.